# 03 — RFM Analysis

Compute R/F/M, distributions raw + log, quintile scoring, traditional segments baseline.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.config import load_settings
from src.rfm import compute_rfm

settings = load_settings(Path.cwd().parent / "config" / "settings.yaml")
silver_path = Path.cwd().parent / settings["paths"]["silver"] / "transactions_clean.parquet"
df = pd.read_parquet(silver_path)
df.shape

(714764, 9)

## Compute RFM

In [2]:
rfm = compute_rfm(df, settings["snapshot_date"])
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,325.0,3.0,65.86
1,12608.0,404.0,1.0,415.79
2,12745.0,486.0,2.0,723.85
3,12746.0,540.0,1.0,230.85
4,12747.0,2.0,26.0,8786.53


In [3]:
rfm.shape, rfm.describe()

((5336, 4),
         Customer ID      Recency    Frequency       Monetary
 count   5336.000000  5336.000000  5336.000000    5336.000000
 mean   15558.976949   202.295352     6.254498    2561.133238
 std     1581.392985   209.992111    11.887700   11141.110049
 min    12346.000000     0.000000     1.000000   -1343.240000
 25%    14190.750000    25.000000     1.000000     322.327500
 50%    15569.500000    97.000000     3.000000     814.455000
 75%    16924.250000   381.000000     7.000000    2099.010000
 max    18287.000000   738.000000   323.000000  578408.640000)

## Quintile scoring

In [4]:
from src.rfm import score_rfm_quintiles

rfm_scored = score_rfm_quintiles(rfm)
rfm_scored.head()

,Customer ID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score
0,12346.0,325.0,3.0,65.86,2,3,1,231
1,12608.0,404.0,1.0,415.79,2,1,2,212
2,12745.0,486.0,2.0,723.85,1,2,3,123
3,12746.0,540.0,1.0,230.85,1,1,1,111
4,12747.0,2.0,26.0,8786.53,5,5,5,555


In [5]:
rfm_scored[["R_score", "F_score", "M_score"]].apply(lambda s: s.value_counts().sort_index())

,R_score,F_score,M_score
1,1067,1068,1068
2,1067,1067,1067
3,1067,1067,1067
4,1067,1067,1067
5,1068,1067,1067


## Traditional segments (baseline)

In [6]:
from src.rfm import assign_traditional_segments

rfm_scored = assign_traditional_segments(rfm_scored)
rfm_scored["Segment"].value_counts()

Segment
Hibernating    1669
Champions      1339
Loyal          1116
At Risk         746
Lost            466
Name: count, dtype: int64

## Save gold/rfm_table.parquet

In [7]:
gold_dir = Path.cwd().parent / settings["paths"]["gold"]
out_path = gold_dir / "rfm_table.parquet"
rfm_scored.to_parquet(out_path, index=False)
print(f"saved: {out_path}")

saved: D:\disertatie v2\Customer Behavior Analysis\data\gold\rfm_table.parquet
